# Módulo 7: Álgebra Lineal Matricial (`numpy.linalg`)

## 🎯 Objetivos de Aprendizaje
Al finalizar este cuaderno, serás capaz de:
1. Calcular el **producto punto** entre vectores y la **multiplicación algebraica de matrices** mediante el operador estándar `@` (`np.matmul`), diferenciándolo tajantemente del producto elemento a elemento (`*`).
2. Comprender por qué no existe una "división directa" entre matrices y resolver su equivalente algebraico mediante la **matriz inversa** (`np.linalg.inv`) y la resolución de **sistemas de ecuaciones lineales** (`np.linalg.solve`).
3. Calcular propiedades fundamentales de vectores y matrices cuadradas: **norma vectorial/matricial** (`np.linalg.norm`) y **determinante** (`np.linalg.det`).
4. Evitar errores clásicos de incompatibilidad de dimensiones internas, matrices singulares (no invertibles) y el uso ineficiente de la inversa para resolver sistemas lineales.

---

## 🗺️ Mapa Mental del Módulo

```text
                    [ Álgebra Lineal en NumPy ]
                                 │
      ┌──────────────────────────┼──────────────────────────┐
      ▼                          ▼                          ▼
1. MULTIPLICACIÓN          2. "DIVISIÓN" Y SISTEMAS   3. PROPIEDADES (linalg)
├── Producto punto (1D)    ├── No existe A / B          ├── Norma: np.linalg.norm()
├── Matricial: A @ B       ├── Inversa: linalg.inv(A)   ├── Determinante: linalg.det()
└── Regla: (M, K) @ (K, N) └── Sistemas: linalg.solve() └── Rango: matrix_rank()

In [1]:
#Importar numpy
import numpy as np

## 7.1 Producto Punto y Multiplicación Matricial (`@` y `np.matmul`)

A diferencia de la aritmética elemento a elemento del Módulo 4 (`A * B`), la **multiplicación matricial** combina filas de la primera matriz con columnas de la segunda mediante sumas de productos escalares:

$$C_{i,j} = \sum_{k=1}^{K} A_{i,k} \cdot B_{k,j}$$

### Regla de Oro de Compatibilidad Dimensional
Para multiplicar una matriz $\mathbf{A}$ de forma $(M, K)$ por una matriz $\mathbf{B}$ de forma $(K, N)$:
1. Las **dimensiones internas ($K$) deben coincidir exactamente** (columnas de $\mathbf{A}$ = filas de $\mathbf{B}$).
2. La matriz resultante $\mathbf{C} = \mathbf{A} @ \mathbf{B}$ tendrá forma **$(M, N)$** (dimensiones externas).
3. **No es conmutativa:** En general, $\mathbf{A} @ \mathbf{B} \neq \mathbf{B} @ \mathbf{A}$.

### Operadores Oficiales
* **`A @ B` (Recomendado, PEP 465):** Operador nativo de Python para producto punto (entre vectores 1D) y multiplicación matricial (entre matrices 2D). Equivale exactamente a `np.matmul(A, B)`.
* **`np.dot(A, B)`:** Función clásica equivalente a `@` en arreglos 1D y 2D (difiere únicamente en tensores de 3D o más).

### Sintaxis Mínima

In [8]:
# 1. Producto punto (escalar) entre dos vectores 1D de 10 elementos
# Ejemplo: Cantidades vendidas de 10 artículos y sus precios unitarios ($)
cantidades = np.array([12, 5, 8, 20, 15, 3, 10, 7, 14, 9])
precios_unitarios = np.array([2.5, 10.0, 4.0, 1.5, 3.0, 25.0, 6.0, 8.5, 2.0, 5.0])
print("Vector cantidades", cantidades)
print("Vector precios", precios_unitarios)

ingreso_total = cantidades @ precios_unitarios
print(f"Ingreso total (producto punto 1D @ 1D): ${ingreso_total:.2f}")

# 2. Multiplicación Matricial 2D: (3x4) @ (4x3) -> (3x3)
# Matriz A (3x4): Horas invertidas por 3 ingenieros (filas) en 4 tareas (columnas)
horas_ingenieros = np.array([
    [10, 5,  8,  2],
    [6,  12, 4,  8],
    [8,  8,  10, 4]
])

# Matriz B (4x3): Costo por hora ($) de cada tarea (filas) en 3 proyectos distintos (columnas)
tarifas_proyecto = np.array([
    [40, 45, 50],
    [35, 38, 40],
    [50, 55, 60],
    [30, 32, 35]
])

print("\nMatriz horas investidas:\n", horas_ingenieros)
print("\nMatriz costo por hora:\n", tarifas_proyecto)

# Cálculo de costos totales por ingeniero y proyecto: (3, 4) @ (4, 3) -> (3, 3)
costo_total_proyectos = horas_ingenieros @ tarifas_proyecto
print("\nMatriz resultante (shape", costo_total_proyectos.shape, "):\n", costo_total_proyectos)

# 3. Verificación de NO conmutatividad en matrices cuadradas (3x3)
M1 = np.array([[2, 1, 0], [1, 3, 2], [0, 2, 1]])
M2 = np.array([[1, 0, 2], [3, 1, 0], [0, 1, 4]])

print("\n¿M1 @ M2 es igual a M2 @ M1?:", np.array_equal(M1 @ M2, M2 @ M1))

Vector cantidades [12  5  8 20 15  3 10  7 14  9]
Vector precios [ 2.5 10.   4.   1.5  3.  25.   6.   8.5  2.   5. ]
Ingreso total (producto punto 1D @ 1D): $454.50

Matriz horas investidas:
 [[10  5  8  2]
 [ 6 12  4  8]
 [ 8  8 10  4]]

Matriz costo por hora:
 [[40 45 50]
 [35 38 40]
 [50 55 60]
 [30 32 35]]

Matriz resultante (shape (3, 3) ):
 [[1035 1144 1250]
 [1100 1202 1300]
 [1220 1342 1460]]

¿M1 @ M2 es igual a M2 @ M1?: False


:::{warning} Trampa Común: Usar `*` en lugar de `@` con matrices cuadradas
Si tienes dos matrices cuadradas $\mathbf{A}$ y $\mathbf{B}$ de $3 \times 3$, escribir `A * B` **no dará ningún error**, pero calculará el producto celda por celda en lugar del producto matricial del álgebra lineal. Utiliza siempre `@` para multiplicación matricial.
:::

:::{note} Multiplicación Matriz-Vector `(M, K) @ (K,)`
NumPy permite multiplicar directamente una matriz 2D de forma `(3, 4)` por un vector 1D de forma `(4,)` mediante `M @ v`. El resultado es un vector 1D de forma `(3,)`. Sin embargo, si los inviertes (`v @ M`), fallará porque `(4,)` no encaja con `(3, 4)` (deberías transponer `M`: `v @ M.T`).
:::

### 🧩 Micro-reto 7.1
1. Crea una matriz `A` de dimensiones $3 \times 5$ con `np.arange(1, 16).reshape(3, 5)` y una matriz `B` de dimensiones $5 \times 2$ con `np.ones((5, 2)) * 2`.
2. Calcula el producto matricial `C = A @ B` e imprime su `shape` y su contenido.
3. Intenta calcular `B @ A` mentalmente antes de probarlo: ¿por qué falla? *(Pista: verifica las dimensiones internas `(5, 2)` vs. `(3, 5)`).*

In [ ]:
# Escribe tu solución aquí:

## 7.2 El Equivalente a la "División" Matricial: Matriz Inversa y `np.linalg.solve`

En álgebra lineal **no existe la operación de división de matrices** ($\mathbf{A} / \mathbf{B}$). En NumPy, `A / B` realiza una división elemento a elemento, **no** una operación algebraica.

Cuando en una ecuación matricial necesitamos "despejar" una matriz o vector incógnita $\mathbf{x}$ en $\mathbf{A} \mathbf{x} = \mathbf{b}$, utilizamos uno de dos mecanismos en `numpy.linalg`:

1. **Matriz Inversa (`np.linalg.inv(A)`):**
   Para una matriz cuadrada $\mathbf{A}$ cuyo determinante sea distinto de cero ($\det(\mathbf{A}) \neq 0$), existe una matriz inversa $\mathbf{A}^{-1}$ tal que $\mathbf{A} @ \mathbf{A}^{-1} = \mathbf{I}$ (matriz identidad). Despejando: $\mathbf{x} = \mathbf{A}^{-1} @ \mathbf{b}$.
2. **Resolución Directa de Sistemas Lineales (`np.linalg.solve(A, b)`):**
   Resuelve el sistema $\mathbf{A} \mathbf{x} = \mathbf{b}$ mediante descomposición LU sin necesidad de invertir $\mathbf{A}$ explícitamente. **Es más rápido y numéricamente más preciso** que calcular `np.linalg.inv(A) @ b`.

### Sintaxis Mínima

In [12]:
# Ejemplo práctico: Sistema de 3 ecuaciones lineales con 3 incógnitas (3x3)
# 3x + 2y -  z =  1
# 2x - 2y + 4z = -2
# -x + 0.5y - z = 0

A = np.array([
    [ 3.0,  2.0, -1.0],
    [ 2.0, -2.0,  4.0],
    [-1.0,  0.5, -1.0]
])

print("Matriz A:\n", A)

b = np.array([1.0, -2.0, 0.0])
print("\nVector b:\n", b)

# Método 1 (Recomendado): Resolución directa con np.linalg.solve
solucion_solve = np.linalg.solve(A, b)
print("\nSolución del sistema (x, y, z) con solve:", solucion_solve)

# Método 2: Usando la matriz inversa explícita (A^-1 @ b)
A_inv = np.linalg.inv(A)
solucion_inv = A_inv @ b
print("Solución del sistema (x, y, z) con inv  :", solucion_inv)

# Comprobación: A @ A^-1 debe dar la Matriz Identidad (I)
identidad_verificada = np.round(A @ A_inv, decimals=10)
print("\nVerificación A @ A_inv = I:\n", identidad_verificada)

# Comprobación del sistema: A @ x debe reconstruir el vector b
print("\nVerificación A @ x = b:", np.allclose(A @ solucion_solve, b))

Matriz A:
 [[ 3.   2.  -1. ]
 [ 2.  -2.   4. ]
 [-1.   0.5 -1. ]]

Vector b:
 [ 1. -2.  0.]

Solución del sistema (x, y, z) con solve: [ 1. -2. -2.]
Solución del sistema (x, y, z) con inv  : [ 1. -2. -2.]

Verificación A @ A_inv = I:
 [[ 1.  0.  0.]
 [-0.  1. -0.]
 [ 0.  0.  1.]]

Verificación A @ x = b: True


:::{note} Reglas y Restricciones de Invertibilidad
Para calcular `np.linalg.inv(A)` o usar `np.linalg.solve(A, b)`:
1. La matriz $\mathbf{A}$ **debe ser estrictamente cuadrada** ($N \times N$).
2. Su determinante debe ser distinto de cero ($\det(\mathbf{A}) \neq 0$, es decir, filas y columnas linealmente independientes). Si una fila es múltiplo de otra, la matriz es **singular** y NumPy lanzará `LinAlgError: Singular matrix`.
:::

:::{tip} Regla de Comparación de Flotantes (`np.allclose`)
Debido a la precisión finita de punto flotante (errores de redondeo de $10^{-16}$), `A @ A_inv` puede contener valores como `1.11e-16` en lugar de `0.0`. Nunca compares matrices de decimales con `==` o `np.array_equal()`; utiliza siempre **`np.allclose(M1, M2)`**.
:::

### 🧩 Micro-reto 7.2
Un taller fabrica tres modelos de piezas metálicas ($x, y, z$) utilizando acero, aluminio y cobre (en kg). La matriz de requerimientos por unidad `materiales` ($3 \times 3$) y la disponibilidad total en bodega `bodega` (vector de 3 elementos) están definidas en la celda inferior:
1. Utiliza `np.linalg.solve(materiales, bodega)` para calcular cuántas unidades exactas de cada modelo ($x, y, z$) se pueden fabricar agotando exactamente toda la bodega.
2. Verifica con `np.allclose(materiales @ unidades, bodega)` que la solución es exacta.

In [14]:
materiales = np.array([
    [2.0, 1.0, 3.0],  # Acero requerido por modelo x, y, z
    [1.0, 3.0, 2.0],  # Aluminio requerido
    [3.0, 2.0, 1.0]   # Cobre requerido
])

bodega = np.array([13.0, 13.0, 10.0])  # Stock total disponible (kg)

# Escribe tu solución aquí:

## 7.3 Propiedades Estructurales: Norma (`np.linalg.norm`) y Determinante (`np.linalg.det`)

Para evaluar magnitudes, distancias geométricas y estabilidad numérica de matrices, `numpy.linalg` provee funciones esenciales:

* **Norma (`np.linalg.norm(x, ord=None, axis=None)`):**
  * En un vector 1D $\mathbf{v} = [v_1, v_2, \dots, v_n]$, calcula por defecto la **norma euclidiana ($L_2$)** o longitud geométrica del vector: $\|\mathbf{v}\|_2 = \sqrt{\sum v_i^2}$.
  * Si especificas `ord=1`, calcula la norma Manhattan ($L_1 = \sum |v_i|$).
  * Si pasas `axis=1` en una matriz 2D, calcula la longitud euclidiana de cada fila independientemente.
* **Determinante (`np.linalg.det(A)`):**
  * Escalar asociado a toda matriz cuadrada ($N \times N$) que mide el factor de escala del volumen de la transformación lineal y diagnostica si la matriz es invertible ($\det(\mathbf{A}) \neq 0$).

### Sintaxis Mínima

In [15]:
# 1. Norma de un vector 1D (ej. vector de desplazamiento o error de predicción)
vector_error = np.array([3.0, -4.0, 0.0, 12.0])

norma_l2 = np.linalg.norm(vector_error)           # Euclidiana: sqrt(9 + 16 + 0 + 144) = 13.0
norma_l1 = np.linalg.norm(vector_error, ord=1)    # Manhattan : |3| + |-4| + |0| + |12| = 19.0
print(f"Norma Euclidiana (L2): {norma_l2} | Norma Manhattan (L1): {norma_l1}")

# 2. Aplicación práctica: Distancia euclidiana entre 4 puntos 3D y el origen usando axis=1
coordenadas_3d = np.array([
    [3.0, 4.0, 0.0],    # Distancia = 5.0
    [1.0, 2.0, 2.0],    # Distancia = 3.0
    [6.0, 8.0, 0.0],    # Distancia = 10.0
    [2.0, 3.0, 6.0]     # Distancia = 7.0
])

distancias_origen = np.linalg.norm(coordenadas_3d, axis=1)
print("\nDistancias euclidianas de cada punto al origen (axis=1):", distancias_origen)

# 3. Determinante de una matriz cuadrada 3x3
matriz_invertible = np.array([
    [4, 2, 1],
    [0, 3, 5],
    [2, 1, 6]
])

matriz_singular = np.array([
    [1, 2, 3],
    [2, 4, 6],  # Fila 1 es exactamente el doble de la Fila 0 (dependencia lineal)
    [7, 8, 9]
])

print(f"\nDeterminante matriz invertible: {np.linalg.det(matriz_invertible):.2f}")
print(f"Determinante matriz singular  : {np.linalg.det(matriz_singular):.2f}")

Norma Euclidiana (L2): 13.0 | Norma Manhattan (L1): 19.0

Distancias euclidianas de cada punto al origen (axis=1): [ 5.  3. 10.  7.]

Determinante matriz invertible: 66.00
Determinante matriz singular  : 0.00


:::{warning} Trampa Común: Normalizar un vector dividiendo por cero
Una operación clásica en Ciencia de Datos es obtener un **vector unitario** (de longitud 1) dividiendo el vector entre su norma: `v_unitario = v / np.linalg.norm(v)`. Si el vector es completamente nulo (`[0, 0, 0]`), su norma es `0.0` y obtendrás `nan`. Verifica siempre que `norma > 0` antes de normalizar.
:::

### 🧩 Micro-reto 7.3
1. Crea un vector `v = np.array([6.0, 8.0, 0.0, 0.0])` y transfórmalo en un **vector unitario** `v_unit` dividiéndolo entre su norma euclidiana (`np.linalg.norm(v)`). Comprueba que `np.linalg.norm(v_unit)` es igual a `1.0`.
2. Dada la matriz `K = np.array([[2, 0, 1], [1, 3, 0], [0, 1, 2]])`, calcula su determinante con `np.linalg.det(K)` para confirmar que es invertible y luego obtén su matriz inversa con `np.linalg.inv(K)`.

In [16]:
# Escribe tu solución aquí:

## ✅ Resumen del Módulo 7
* Utiliza **`@`** (`np.matmul`) para multiplicación algebraica de matrices y producto punto de vectores; reserva **`*`** exclusivamente para operaciones elemento a elemento.
* En la multiplicación `(M, K) @ (K, N)`, las dimensiones internas ($K$) deben coincidir y el resultado tiene forma `(M, N)`.
* En álgebra lineal no existe `A / B`; para resolver sistemas $\mathbf{A}\mathbf{x} = \mathbf{b}$ emplea **`np.linalg.solve(A, b)`** en lugar de calcular manualmente la inversa.
* Usa **`np.allclose()`** al verificar igualdades entre matrices resultantes de operaciones en punto flotante.